# threading

对应 `stdlib.md`：线程。

笔记本在 `python_stdlibs/threading/qa.ipynb`。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录，即 python_stdlibs/threading。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "threading" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "python_stdlibs" / "threading"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT

PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/python_stdlibs/threading')

## 1. 起一个线程，等它结束

`task` 已经写好，会打印当前线程的名字。

起一个名为 `worker-1` 的线程跑它。等这个线程结束后，主线程再打印 `main done`。

In [ ]:
import threading

def task() -> None:
    print(f"{threading.current_thread().name} 在跑")

# 作答

thread = threading.Thread(target=task, name="worker-1")
thread.start()

thread.join()

print("main done")
#评阅
# 对。start 起了线程，join 等它结束，main done 排在线程那行之后。名字也传对了。

#参考答案
# thread = threading.Thread(target=task, name="worker-1")
# thread.start()
# thread.join()
# print("main done")

worker-1 在跑
main done


## 2. 同时跑两个线程

`worker(name, seconds)` 会等 `seconds` 秒，然后打印 `name 完成`。

同时起两个线程，参数分别是 `("A", 0.5)` 和 `("B", 0.05)`，两个都结束后再打印 `all done`。

In [4]:
import threading
import time

def worker(name: str, seconds: float) -> None:
    time.sleep(seconds)
    print(f"{name} 完成")

# 作答

start_time = time.time()
threads = list(
    threading.Thread(target=worker, args=params) for params in
        [("A", 0.5), ("B", 0.05)]
    )

for thread in threads:
    thread.start()

for thread in threads:
    thread.join()

print(f"all done {time.time() - start_time}")
#评阅
# 对。0.504 秒贴近较慢的那一个（0.5s），串行会是 0.55s——这个计时正好把「两个线程在同时跑」证出来了，
# 是这道题最该看到的证据。args 传元组，Thread 会自动展开成位置参数。

#参考答案
# threads = [threading.Thread(target=worker, args=("A", 0.5)),
#            threading.Thread(target=worker, args=("B", 0.05))]
# for t in threads: t.start()
# for t in threads: t.join()
# print("all done")

B 完成
A 完成
all done 0.504845142364502


## 3. 把线程的结果拿回来

`compute(name, seconds)` 会等 `seconds` 秒，然后返回一个字符串。

起两个线程跑 `compute("A", 0.1)` 和 `compute("B", 0.2)`。把各自的返回值收进一个字典，键是 `name`，两个都结束后打印这个字典。

In [ ]:
import threading
import time

def compute(name: str, seconds: float) -> str:
    time.sleep(seconds)
    return f"{name} 的结果"

# 作答

THREAD_ARGS = [
    ("A", 0.1),
    ("B", 0.2),
]

results = {}

def thread_worker(name, seconds):
    results[name] = compute(name, seconds)

threads = [
    threading.Thread(target=thread_worker, args=args)
    for args in THREAD_ARGS
]
# print(type(threads[0]).__name__)

for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

print(results)
#评阅
# 对。线程函数没有返回值，结果只能由线程自己写进共享容器；两个都 join 完再读 results 是安全的。
# 那句注释掉的 print(type(...)) 可以删了，下面的输出里还留着它当时的 Thread。

#参考答案
# results = {}
# def run(name, seconds):
#     results[name] = compute(name, seconds)
# threads = [threading.Thread(target=run, args=("A", 0.1)),
#            threading.Thread(target=run, args=("B", 0.2))]
# for t in threads: t.start()
# for t in threads: t.join()
# print(results)

Thread
{'A': 'A 的结果', 'B': 'B 的结果'}


## 4. 不加锁会丢更新

起两个线程，每个都做 200 次这个动作：读出 `total`，等 `0.001` 秒，再把 `total + 1` 写回去。两个都结束后打印 `total`。

然后加一把 `threading.Lock`，让「读出 + 写回」中间不能被打断，再跑一遍。两次的 `total` 一样吗？

In [21]:
import threading
import time

total = 0

# 作答

def worker(lock: threading.Lock | None) -> None:
    global total
    for _ in range(200):
        if lock is not None:
            with lock:
                value = total
                time.sleep(0.001)
                total = value + 1
        else:
            value = total
            time.sleep(0.001)
            total = value + 1

threads = list(threading.Thread(target=worker, args=(None,)) for _ in range(2))

for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

print(total)

total = 0
lock = threading.Lock()

threads = list(threading.Thread(target=worker, args=(lock,)) for _ in range(2))

for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

print(total)
#评阅
# 对。200 和 400 正是要的对比：不加锁时两个线程读到同一个旧值、又各自写回同一个新值，那一轮白做，
# 所以只前进一半；加锁把「读 - 等 - 写」变成临界区，锁住之后才到 400。
# 两个分支把读-睡-写抄了两遍，可以用 contextlib.nullcontext 让没锁那次也走 with，省掉重复。

#参考答案
# lock = threading.Lock()
# def worker():
#     global total
#     for _ in range(200):
#         with lock:
#             value = total
#             time.sleep(0.001)
#             total = value + 1
# total = 0
# ts = [threading.Thread(target=worker) for _ in range(2)]
# for t in ts: t.start()
# for t in ts: t.join()
# print(total)   # 400

200
400


## 5. 等另一个线程发信号

起一个线程，它先打印 `waiting`，然后停在那里等信号，收到信号之后才打印 `got it`。

主线程等 0.1 秒后发信号（用 `threading.Event`），再等那个线程结束。

In [26]:
import threading
import time

# 作答

def worker(event: threading.Event) -> None:
    print("waiting")
    event.wait()
    print("got it")

event = threading.Event()

thread = threading.Thread(target=worker, args=(event,))
thread.start()

time.sleep(2)
event.set()

thread.join()
#评阅
# 对。waiting 之后卡在 event.wait()，主线程 set 之后才轮到 got it，顺序就是这个意思。
# 不过题目要的是等 0.1 秒，你写了 2 秒——机制一样，只是白等 1.9 秒。

#参考答案
# event = threading.Event()
# def worker():
#     print("waiting")
#     event.wait()
#     print("got it")
# t = threading.Thread(target=worker)
# t.start()
# time.sleep(0.1)
# event.set()
# t.join()

waiting
got it


## 6. 限制同时运行的数量

`job(i)` 已经写好，进来先打印，再等 0.2 秒。

起 5 个线程跑 `job(0)` 到 `job(4)`。用 `threading.Semaphore(2)` 限制：最多只能有 2 个线程同时待在 `job` 里面。全部结束后打印 `end`。

In [45]:
import threading
import time

def job(i: int) -> None:
    print(f"job {i} 开始")
    time.sleep(1)

# 作答

sem = threading.Semaphore(2)

threads = []

def job_wrapped(i):
    with sem:
        job(i)

for i in range(5):
    t = threading.Thread(target=job_wrapped, args=(i,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print("end")
#评阅
# with sem 包住 job(i)，正是这道题要的写法，代码是对的。
# 但这份输出证明不了限流真的生效：print 的顺序在限流和不限流下都是 0、1、2、3、4，
# 因为 Jupyter 把 stdout 收完才一次性显示，看不到每个 job 是什么时候进去的。
# 实测量化：sleep(1) x 5 个任务，限流 2 个要跑 3 轮约 3.0 秒，去掉 sem 只要 1.0 秒（3.01s vs 1.00s）。
# 另外你把前置代码的 sleep(0.2) 改成了 1，前置代码本来不用动。

#参考答案
# 想看出限流生效，加上耗时（或记录同时在 job 里的峰值）：
# sem = threading.Semaphore(2)
# def run(i):
#     with sem:
#         job(i)
# threads = [threading.Thread(target=run, args=(i,)) for i in range(5)]
# t0 = time.perf_counter()
# for t in threads: t.start()
# for t in threads: t.join()
# print(f"end {time.perf_counter() - t0:.2f}s")   # 约 3 秒；去掉 sem 只剩 1 秒

job 0 开始
job 1 开始
job 2 开始
job 3 开始
job 4 开始
end
